# OrienterNet: Prediction Statistics & Analysis
### Statistics per cell with S2 grid visualization

## Cell 1: Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
from scipy.spatial.distance import cdist
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ All imports successful")

## Cell 2: Load Data

In [ ]:
# Load results
results_file = 'orienternet_results.csv'
df = pd.read_csv(results_file)

print(f"Loaded {len(df)} predictions")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst rows:")
print(df.head())

# Load S2 cells
s2_cells = gpd.read_file('data/cells/bern_13.gpkg')
print(f"\nLoaded {len(s2_cells)} S2 cells")
print(f"S2 cells CRS: {s2_cells.crs}")

## Cell 3: Calculate Geodetic Distances (Haversine - WGS84)

In [ ]:
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate great circle distance between two points 
    on the earth (specified in decimal degrees)
    Returns distance in meters
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371000  # Radius of earth in meters
    return c * r

# Calculate distances
df['distance_m'] = df.apply(
    lambda row: haversine(
        row['pred_lon'], row['pred_lat'],
        row['true_lon'], row['true_lat']
    ), axis=1
)

print(f"Distance Statistics (meters):")
print(df['distance_m'].describe())
print(f"\nMedian distance: {df['distance_m'].median():.2f} m")

## Cell 4: Calculate Yaw Errors

In [ ]:
def normalize_angle(angle):
    """Normalize angle to [-180, 180]"""
    while angle > 180:
        angle -= 360
    while angle < -180:
        angle += 360
    return angle

# Calculate yaw error (signed and absolute)
df['yaw_error_signed'] = df.apply(
    lambda row: normalize_angle(row['pred_yaw'] - row['true_yaw']),
    axis=1
)
df['yaw_error_abs'] = df['yaw_error_signed'].abs()

print(f"Yaw Error Statistics (degrees):")
print(f"  Min: {df['yaw_error_abs'].min():.2f}°")
print(f"  Max: {df['yaw_error_abs'].max():.2f}°")
print(f"  Mean: {df['yaw_error_abs'].mean():.2f}°")
print(f"  Median: {df['yaw_error_abs'].median():.2f}°")
print(f"  Std: {df['yaw_error_abs'].std():.2f}°")

## Cell 5: Loss Functions

In [ ]:
# Calculate different loss metrics
mae_distance = df['distance_m'].mean()
rmse_distance = np.sqrt((df['distance_m']**2).mean())
mae_yaw = df['yaw_error_abs'].mean()
rmse_yaw = np.sqrt((df['yaw_error_abs']**2).mean())

# Normalized combined loss (equal weight)
# Normalize by median values for fair comparison
median_dist = df['distance_m'].median()
median_yaw = df['yaw_error_abs'].median()

df['loss_combined'] = (
    (df['distance_m'] / median_dist) + 
    (df['yaw_error_abs'] / median_yaw)
) / 2

print("Loss Metrics:")
print(f"  Distance MAE: {mae_distance:.2f} m")
print(f"  Distance RMSE: {rmse_distance:.2f} m")
print(f"  Yaw MAE: {mae_yaw:.2f}°")
print(f"  Yaw RMSE: {rmse_yaw:.2f}°")
print(f"  Combined Loss Mean: {df['loss_combined'].mean():.4f}")
print(f"  Combined Loss Median: {df['loss_combined'].median():.4f}")

## Cell 6: Distance Error Histogram (Linear)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Linear scale
ax1.hist(df['distance_m'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(df['distance_m'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["distance_m"].mean():.1f}m')
ax1.axvline(df['distance_m'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["distance_m"].median():.1f}m')
ax1.set_xlabel('Distance Error (meters)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distance Error Distribution (Linear Scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log scale
ax2.hist(df['distance_m'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax2.set_yscale('log')
ax2.axvline(df['distance_m'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["distance_m"].mean():.1f}m')
ax2.axvline(df['distance_m'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["distance_m"].median():.1f}m')
ax2.set_xlabel('Distance Error (meters)')
ax2.set_ylabel('Frequency (log scale)')
ax2.set_title('Distance Error Distribution (Log Scale)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 7: Yaw Error Histogram

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Signed yaw errors
ax1.hist(df['yaw_error_signed'], bins=50, color='coral', edgecolor='black', alpha=0.7)
ax1.axvline(df['yaw_error_signed'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["yaw_error_signed"].mean():.1f}°')
ax1.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax1.set_xlabel('Yaw Error Signed (degrees)')
ax1.set_ylabel('Frequency')
ax1.set_title('Signed Yaw Error Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Absolute yaw errors
ax2.hist(df['yaw_error_abs'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax2.axvline(df['yaw_error_abs'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["yaw_error_abs"].mean():.1f}°')
ax2.axvline(df['yaw_error_abs'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["yaw_error_abs"].median():.1f}°')
ax2.set_xlabel('Yaw Error Absolute (degrees)')
ax2.set_ylabel('Frequency')
ax2.set_title('Absolute Yaw Error Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 8: Percentile Analysis

In [ ]:
percentiles = [50, 75, 90, 95, 99]

print("Distance Error Percentiles:")
for p in percentiles:
    val = np.percentile(df['distance_m'], p)
    print(f"  {p}th percentile: {val:.2f} m")

print("\nYaw Error Percentiles (absolute):")
for p in percentiles:
    val = np.percentile(df['yaw_error_abs'], p)
    print(f"  {p}th percentile: {val:.2f}°")

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

perc_vals_dist = [np.percentile(df['distance_m'], p) for p in percentiles]
ax1.bar([str(p) for p in percentiles], perc_vals_dist, color='steelblue')
ax1.set_xlabel('Percentile')
ax1.set_ylabel('Distance (meters)')
ax1.set_title('Distance Error Percentiles')
ax1.grid(True, alpha=0.3, axis='y')

perc_vals_yaw = [np.percentile(df['yaw_error_abs'], p) for p in percentiles]
ax2.bar([str(p) for p in percentiles], perc_vals_yaw, color='coral')
ax2.set_xlabel('Percentile')
ax2.set_ylabel('Yaw Error (degrees)')
ax2.set_title('Yaw Error Percentiles')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Cell 9: Cumulative Distribution Function (CDF)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distance CDF
sorted_dist = np.sort(df['distance_m'])
cdf_dist = np.arange(1, len(sorted_dist)+1) / len(sorted_dist)
ax1.plot(sorted_dist, cdf_dist, linewidth=2, color='steelblue')
ax1.set_xlabel('Distance Error (meters)')
ax1.set_ylabel('Cumulative Probability')
ax1.set_title('Distance Error CDF')
ax1.grid(True, alpha=0.3)
ax1.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50%')
ax1.axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='90%')
ax1.legend()

# Yaw Error CDF
sorted_yaw = np.sort(df['yaw_error_abs'])
cdf_yaw = np.arange(1, len(sorted_yaw)+1) / len(sorted_yaw)
ax2.plot(sorted_yaw, cdf_yaw, linewidth=2, color='coral')
ax2.set_xlabel('Yaw Error (degrees)')
ax2.set_ylabel('Cumulative Probability')
ax2.set_title('Yaw Error CDF')
ax2.grid(True, alpha=0.3)
ax2.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50%')
ax2.axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='90%')
ax2.legend()

plt.tight_layout()
plt.show()

## Cell 10: Statistical Summary

In [ ]:
print("="*60)
print("DISTANCE ERROR STATISTICS (meters)")
print("="*60)
print(f"  Count: {len(df)}")
print(f"  Mean: {df['distance_m'].mean():.2f} m")
print(f"  Median: {df['distance_m'].median():.2f} m")
print(f"  Std Dev: {df['distance_m'].std():.2f} m")
print(f"  Min: {df['distance_m'].min():.2f} m")
print(f"  Max: {df['distance_m'].max():.2f} m")
print(f"  Skewness: {stats.skew(df['distance_m']):.4f}")
print(f"  Kurtosis: {stats.kurtosis(df['distance_m']):.4f}")
print(f"  Q1 (25%): {df['distance_m'].quantile(0.25):.2f} m")
print(f"  Q3 (75%): {df['distance_m'].quantile(0.75):.2f} m")

print("\n" + "="*60)
print("YAW ERROR STATISTICS (degrees)")
print("="*60)
print(f"  Count: {len(df)}")
print(f"  Mean: {df['yaw_error_abs'].mean():.2f}°")
print(f"  Median: {df['yaw_error_abs'].median():.2f}°")
print(f"  Std Dev: {df['yaw_error_abs'].std():.2f}°")
print(f"  Min: {df['yaw_error_abs'].min():.2f}°")
print(f"  Max: {df['yaw_error_abs'].max():.2f}°")
print(f"  Skewness: {stats.skew(df['yaw_error_abs']):.4f}")
print(f"  Kurtosis: {stats.kurtosis(df['yaw_error_abs']):.4f}")
print(f"  Q1 (25%): {df['yaw_error_abs'].quantile(0.25):.2f}°")
print(f"  Q3 (75%): {df['yaw_error_abs'].quantile(0.75):.2f}°")

## Cell 11: Correlation Analysis

In [ ]:
correlation = df['distance_m'].corr(df['yaw_error_abs'])
print(f"Correlation between Distance and Yaw Error: {correlation:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['distance_m'], df['yaw_error_abs'], alpha=0.5, s=20)
ax.set_xlabel('Distance Error (meters)')
ax.set_ylabel('Yaw Error (degrees)')
ax.set_title(f'Distance vs Yaw Error Correlation (r={correlation:.4f})')
ax.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(df['distance_m'], df['yaw_error_abs'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['distance_m'].min(), df['distance_m'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend')
ax.legend()

plt.tight_layout()
plt.show()

## Cell 12: Map Center Calculation

In [ ]:
# Calculate map center from S2 cells
s2_bounds = s2_cells.total_bounds  # (minx, miny, maxx, maxy)
map_center_lat = (s2_bounds[1] + s2_bounds[3]) / 2
map_center_lon = (s2_bounds[0] + s2_bounds[2]) / 2

print(f"Map Center: ({map_center_lat:.4f}, {map_center_lon:.4f})")
print(f"Bounds: {s2_bounds}")

## Cell 13: S2 Cell Visualization - Mean Errors per Cell (Distance & Yaw)

In [ ]:
from shapely.geometry import Point

# Ensure DataFrames are in the same CRS
if s2_cells.crs != 'EPSG:4326':
    s2_cells = s2_cells.to_crs('EPSG:4326')

# Create GeoDataFrame for predictions
geometry = [Point(xy) for xy in zip(df['pred_lon'], df['pred_lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Spatial join to find which S2 cell each prediction belongs to
points_in_cells = gpd.sjoin(gdf, s2_cells, how='left', predicate='within')

# Calculate mean errors per S2 cell
cell_stats = points_in_cells.groupby('index_right').agg({
    'distance_m': ['mean', 'std', 'count'],
    'yaw_error_abs': ['mean', 'std'],
    'yaw_error_signed': 'mean'
}).reset_index()

# Flatten column names
cell_stats.columns = ['cell_idx', 'dist_mean', 'dist_std', 'count', 'yaw_mean', 'yaw_std', 'yaw_signed_mean']

# Merge back to get geometry
s2_stats = s2_cells.reset_index().rename(columns={'index': 'cell_idx'})
s2_stats = s2_stats.merge(cell_stats, on='cell_idx', how='left')

# Fill NaN (cells with no predictions) with 0 for visualization
s2_stats['dist_mean'] = s2_stats['dist_mean'].fillna(0)
s2_stats['yaw_mean'] = s2_stats['yaw_mean'].fillna(0)
s2_stats['count'] = s2_stats['count'].fillna(0)

print(f"Total S2 cells: {len(s2_stats)}")
print(f"Cells with predictions: {(s2_stats['count'] > 0).sum()}")
print(f"\nMean distance error across cells: {s2_stats[s2_stats['count'] > 0]['dist_mean'].mean():.2f} m")
print(f"Mean yaw error across cells: {s2_stats[s2_stats['count'] > 0]['yaw_mean'].mean():.2f}°")

## Cell 14: Choropleth Map - Distance Errors per Cell

In [ ]:
# Create distance error choropleth
m_dist = folium.Map(
    location=[map_center_lat, map_center_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Normalize distance errors for color scale (only cells with data)
max_dist = s2_stats[s2_stats['count'] > 0]['dist_mean'].max()
min_dist = s2_stats[s2_stats['count'] > 0]['dist_mean'].min()

for idx, row in s2_stats.iterrows():
    if row['count'] > 0:  # Only draw cells with data
        # Normalize value to [0, 1]
        norm_value = (row['dist_mean'] - min_dist) / (max_dist - min_dist) if max_dist > min_dist else 0
        
        # Color map: green (low) -> yellow -> orange -> red (high)
        if norm_value < 0.25:
            color = '#2ecc71'  # Green
        elif norm_value < 0.5:
            color = '#f1c40f'  # Yellow
        elif norm_value < 0.75:
            color = '#ff9500'  # Orange
        else:
            color = '#e74c3c'  # Red
        
        # Convert geometry to GeoJSON
        if row.geometry.geom_type == 'Polygon':
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda x, color=color: {
                    'fillColor': color,
                    'color': 'black',
                    'weight': 0.5,
                    'fillOpacity': 0.7
                },
                popup=folium.Popup(
                    f"<b>Mean Distance Error:</b> {row['dist_mean']:.2f} m<br>"
                    f"<b>Std Dev:</b> {row['dist_std']:.2f} m<br>"
                    f"<b>Predictions:</b> {int(row['count'])}<br>"
                    f"<b>Mean Yaw Error:</b> {row['yaw_mean']:.2f}°",
                    max_width=250
                )
            ).add_to(m_dist)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 220px; height: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px">
     <p><b>Mean Distance Error (m)</b></p>
     <p><i class="fa fa-square" style="color:#2ecc71"></i> Low error</p>
     <p><i class="fa fa-square" style="color:#f1c40f"></i> Low-medium</p>
     <p><i class="fa fa-square" style="color:#ff9500"></i> Medium-high</p>
     <p><i class="fa fa-square" style="color:#e74c3c"></i> High error</p>
     <p><b>Range:</b> {:.2f} - {:.2f} m</p>
</div>
'''.format(min_dist, max_dist)
m_dist.get_root().html.add_child(folium.Element(legend_html))

m_dist.save('map_distance_errors_per_cell.html')
print("✓ Map saved as 'map_distance_errors_per_cell.html'")
m_dist

## Cell 15: Choropleth Map - Yaw Errors per Cell

In [ ]:
# Create yaw error choropleth
m_yaw = folium.Map(
    location=[map_center_lat, map_center_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Normalize yaw errors for color scale (only cells with data)
max_yaw = s2_stats[s2_stats['count'] > 0]['yaw_mean'].max()
min_yaw = s2_stats[s2_stats['count'] > 0]['yaw_mean'].min()

for idx, row in s2_stats.iterrows():
    if row['count'] > 0:  # Only draw cells with data
        # Normalize value to [0, 1]
        norm_value = (row['yaw_mean'] - min_yaw) / (max_yaw - min_yaw) if max_yaw > min_yaw else 0
        
        # Color map: green (low) -> yellow -> orange -> red (high)
        if norm_value < 0.25:
            color = '#2ecc71'  # Green
        elif norm_value < 0.5:
            color = '#f1c40f'  # Yellow
        elif norm_value < 0.75:
            color = '#ff9500'  # Orange
        else:
            color = '#e74c3c'  # Red
        
        # Convert geometry to GeoJSON
        if row.geometry.geom_type == 'Polygon':
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda x, color=color: {
                    'fillColor': color,
                    'color': 'black',
                    'weight': 0.5,
                    'fillOpacity': 0.7
                },
                popup=folium.Popup(
                    f"<b>Mean Yaw Error:</b> {row['yaw_mean']:.2f}°<br>"
                    f"<b>Std Dev:</b> {row['yaw_std']:.2f}°<br>"
                    f"<b>Signed Mean:</b> {row['yaw_signed_mean']:.2f}°<br>"
                    f"<b>Predictions:</b> {int(row['count'])}<br>"
                    f"<b>Mean Distance Error:</b> {row['dist_mean']:.2f} m",
                    max_width=250
                )
            ).add_to(m_yaw)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 220px; height: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px">
     <p><b>Mean Yaw Error (degrees)</b></p>
     <p><i class="fa fa-square" style="color:#2ecc71"></i> Low error</p>
     <p><i class="fa fa-square" style="color:#f1c40f"></i> Low-medium</p>
     <p><i class="fa fa-square" style="color:#ff9500"></i> Medium-high</p>
     <p><i class="fa fa-square" style="color:#e74c3c"></i> High error</p>
     <p><b>Range:</b> {:.2f} - {:.2f}°</p>
</div>
'''.format(min_yaw, max_yaw)
m_yaw.get_root().html.add_child(folium.Element(legend_html))

m_yaw.save('map_yaw_errors_per_cell.html')
print("✓ Map saved as 'map_yaw_errors_per_cell.html'")
m_yaw

## Cell 16: Box Plots - Error Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Distance box plot
bp1 = ax1.boxplot([df['distance_m']], vert=True, patch_artist=True)
bp1['boxes'][0].set_facecolor('steelblue')
ax1.set_ylabel('Distance Error (meters)')
ax1.set_title('Distance Error Box Plot')
ax1.set_xticklabels(['Distance'])
ax1.grid(True, alpha=0.3, axis='y')

# Yaw box plot
bp2 = ax2.boxplot([df['yaw_error_abs']], vert=True, patch_artist=True)
bp2['boxes'][0].set_facecolor('coral')
ax2.set_ylabel('Yaw Error (degrees)')
ax2.set_title('Yaw Error Box Plot')
ax2.set_xticklabels(['Yaw'])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Box Plot Statistics:")
print(f"\nDistance Error:")
print(f"  Q1: {df['distance_m'].quantile(0.25):.2f} m")
print(f"  Median: {df['distance_m'].median():.2f} m")
print(f"  Q3: {df['distance_m'].quantile(0.75):.2f} m")
print(f"  IQR: {df['distance_m'].quantile(0.75) - df['distance_m'].quantile(0.25):.2f} m")

print(f"\nYaw Error:")
print(f"  Q1: {df['yaw_error_abs'].quantile(0.25):.2f}°")
print(f"  Median: {df['yaw_error_abs'].median():.2f}°")
print(f"  Q3: {df['yaw_error_abs'].quantile(0.75):.2f}°")
print(f"  IQR: {df['yaw_error_abs'].quantile(0.75) - df['yaw_error_abs'].quantile(0.25):.2f}°")

## Cell 17: Regional Analysis - Errors by Geographic Bins

In [ ]:
# Create 5x5 degree latitude/longitude bins
df['lat_bin'] = pd.cut(df['true_lat'], bins=6)
df['lon_bin'] = pd.cut(df['true_lon'], bins=6)

# Calculate stats per region
regional_stats = df.groupby(['lat_bin', 'lon_bin']).agg({
    'distance_m': ['mean', 'std', 'count'],
    'yaw_error_abs': ['mean', 'std']
}).round(2)

print("Regional Statistics (by Lat/Lon bins):")
print(regional_stats)

# Heatmap of mean distance error by region
pivot_dist = df.pivot_table(values='distance_m', index='lat_bin', columns='lon_bin', aggfunc='mean')
pivot_yaw = df.pivot_table(values='yaw_error_abs', index='lat_bin', columns='lon_bin', aggfunc='mean')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot_dist, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax1, cbar_kws={'label': 'Mean Distance (m)'})
ax1.set_title('Mean Distance Error by Region')
ax1.set_xlabel('Longitude Bin')
ax1.set_ylabel('Latitude Bin')

sns.heatmap(pivot_yaw, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax2, cbar_kws={'label': 'Mean Yaw (°)'})
ax2.set_title('Mean Yaw Error by Region')
ax2.set_xlabel('Longitude Bin')
ax2.set_ylabel('Latitude Bin')

plt.tight_layout()
plt.show()

## Cell 18: Error Categories Pie Chart

In [ ]:
# Categorize predictions
def categorize_error(dist, yaw):
    """Categorize based on distance and yaw error thresholds"""
    dist_threshold = df['distance_m'].median()
    yaw_threshold = df['yaw_error_abs'].median()
    
    if dist < dist_threshold and yaw < yaw_threshold:
        return 'Excellent'
    elif dist < dist_threshold * 1.5 and yaw < yaw_threshold * 1.5:
        return 'Good'
    elif dist < dist_threshold * 2.5 or yaw < yaw_threshold * 2.5:
        return 'Acceptable'
    else:
        return 'Poor'

df['error_category'] = df.apply(lambda x: categorize_error(x['distance_m'], x['yaw_error_abs']), axis=1)

category_counts = df['error_category'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
ax1.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Prediction Error Categories')

# Bar chart
category_counts.plot(kind='bar', ax=ax2, color=colors)
ax2.set_title('Count by Error Category')
ax2.set_xlabel('Category')
ax2.set_ylabel('Count')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nError Category Distribution:")
for cat in ['Excellent', 'Good', 'Acceptable', 'Poor']:
    count = (df['error_category'] == cat).sum()
    pct = 100 * count / len(df)
    print(f"  {cat}: {count} ({pct:.1f}%)")

## Cell 19: Quality Score Analysis

In [ ]:
# Create combined quality score
# Normalize errors to 0-100 scale (100 is perfect)
max_dist = df['distance_m'].quantile(0.95)
max_yaw = df['yaw_error_abs'].quantile(0.95)

df['quality_score'] = 100 * (
    0.5 * (1 - np.minimum(df['distance_m'] / max_dist, 1)) +
    0.5 * (1 - np.minimum(df['yaw_error_abs'] / max_yaw, 1))
)

print("Quality Score Statistics:")
print(f"  Mean: {df['quality_score'].mean():.2f}")
print(f"  Median: {df['quality_score'].median():.2f}")
print(f"  Std: {df['quality_score'].std():.2f}")
print(f"  Min: {df['quality_score'].min():.2f}")
print(f"  Max: {df['quality_score'].max():.2f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['quality_score'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(df['quality_score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["quality_score"].mean():.1f}')
ax.axvline(df['quality_score'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["quality_score"].median():.1f}')
ax.set_xlabel('Quality Score (0-100)')
ax.set_ylabel('Frequency')
ax.set_title('Combined Quality Score Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 20: Export Results

In [ ]:
# Export summary statistics
summary_stats = {
    'Metric': [
        'Distance MAE (m)',
        'Distance RMSE (m)',
        'Distance Median (m)',
        'Distance 90th percentile (m)',
        'Yaw MAE (°)',
        'Yaw RMSE (°)',
        'Yaw Median (°)',
        'Yaw 90th percentile (°)',
        'Total Predictions',
        'Mean Quality Score'
    ],
    'Value': [
        f"{df['distance_m'].mean():.2f}",
        f"{np.sqrt((df['distance_m']**2).mean()):.2f}",
        f"{df['distance_m'].median():.2f}",
        f"{np.percentile(df['distance_m'], 90):.2f}",
        f"{df['yaw_error_abs'].mean():.2f}",
        f"{np.sqrt((df['yaw_error_abs']**2).mean()):.2f}",
        f"{df['yaw_error_abs'].median():.2f}",
        f"{np.percentile(df['yaw_error_abs'], 90):.2f}",
        f"{len(df)}",
        f"{df['quality_score'].mean():.2f}"
    ]
}

summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv('analysis_summary.csv', index=False)
print("✓ Summary exported to 'analysis_summary.csv'")
print(summary_df)

# Export detailed results with errors
export_cols = ['pred_lat', 'pred_lon', 'true_lat', 'true_lon', 'pred_yaw', 'true_yaw', 
               'distance_m', 'yaw_error_signed', 'yaw_error_abs', 'quality_score', 'error_category']
df[export_cols].to_csv('predictions_with_errors.csv', index=False)
print("✓ Detailed results exported to 'predictions_with_errors.csv'")

# Export cell statistics
cell_export = s2_stats[s2_stats['count'] > 0][['cell_idx', 'dist_mean', 'dist_std', 'yaw_mean', 'yaw_std', 'count']]
cell_export.to_csv('s2_cell_statistics.csv', index=False)
print("✓ Cell statistics exported to 's2_cell_statistics.csv'")

print("\n✓ All analysis complete!")

## Cell 21: Summary & Recommendations

In [ ]:
print("="*70)
print(" ANALYSIS SUMMARY & RECOMMENDATIONS")
print("="*70)

print("\n📍 GEOGRAPHIC DISTRIBUTION:")
cells_with_data = (s2_stats['count'] > 0).sum()
print(f"   - Predictions distributed across {cells_with_data} S2 cells (level 13)")
print(f"   - Total predictions: {len(df)}")
print(f"   - Average per cell: {len(df)/cells_with_data:.1f}")

print("\n📊 DISTANCE ERROR:")
print(f"   - Mean: {df['distance_m'].mean():.2f} m")
print(f"   - Median: {df['distance_m'].median():.2f} m")
print(f"   - 90th percentile: {np.percentile(df['distance_m'], 90):.2f} m")
print(f"   - Best cell: {s2_stats[s2_stats['count'] > 0]['dist_mean'].min():.2f} m")
print(f"   - Worst cell: {s2_stats[s2_stats['count'] > 0]['dist_mean'].max():.2f} m")

print("\n🔄 ORIENTATION (YAW) ERROR:")
print(f"   - Mean: {df['yaw_error_abs'].mean():.2f}°")
print(f"   - Median: {df['yaw_error_abs'].median():.2f}°")
print(f"   - 90th percentile: {np.percentile(df['yaw_error_abs'], 90):.2f}°")
print(f"   - Best cell: {s2_stats[s2_stats['count'] > 0]['yaw_mean'].min():.2f}°")
print(f"   - Worst cell: {s2_stats[s2_stats['count'] > 0]['yaw_mean'].max():.2f}°")

print("\n⭐ QUALITY ASSESSMENT:")
excellent_pct = 100 * (df['error_category'] == 'Excellent').sum() / len(df)
good_pct = 100 * (df['error_category'] == 'Good').sum() / len(df)
acceptable_pct = 100 * (df['error_category'] == 'Acceptable').sum() / len(df)
poor_pct = 100 * (df['error_category'] == 'Poor').sum() / len(df)

print(f"   - Excellent: {excellent_pct:.1f}% ({(df['error_category'] == 'Excellent').sum()})")
print(f"   - Good: {good_pct:.1f}% ({(df['error_category'] == 'Good').sum()})")
print(f"   - Acceptable: {acceptable_pct:.1f}% ({(df['error_category'] == 'Acceptable').sum()})")
print(f"   - Poor: {poor_pct:.1f}% ({(df['error_category'] == 'Poor').sum()})")
print(f"   - Mean Quality Score: {df['quality_score'].mean():.2f}/100")

print("\n🗺️  GENERATED MAPS:")
print("   - map_distance_errors_per_cell.html (distance errors by S2 cell)")
print("   - map_yaw_errors_per_cell.html (orientation errors by S2 cell)")

print("\n📁 EXPORTED FILES:")
print("   - analysis_summary.csv (key metrics)")
print("   - predictions_with_errors.csv (detailed predictions)")
print("   - s2_cell_statistics.csv (per-cell statistics)")

print("\n💡 RECOMMENDATIONS:")
if poor_pct > 10:
    print("   ⚠️  High proportion of poor predictions. Consider model retraining or data augmentation.")
if np.percentile(df['distance_m'], 90) > 1000:
    print("   ⚠️  Some predictions have very large localization errors (>1000m).")
if abs(df['yaw_error_signed'].mean()) > 5:
    print("   ⚠️  Orientation bias detected (mean yaw error != 0).")
if correlation > 0.3:
    print(f"   ℹ️  Correlation between distance and yaw errors: {correlation:.3f} (moderate).")

print("\n" + "="*70)